In [13]:
import numpy as np
import nn_fncs
mat_data = nn_fncs.read_mat_workspace('Thrust_data.mat')
nn_in = mat_data.get('nn_in')
nn_out = mat_data.get('nn_out')
# Get every 5th data sample
nn_in = nn_in[:, ::5, :]
nn_out = nn_out[:, ::5, :]
# remove zero columns in nn_out
# nn_out1 = nn_out[:, :, ~np.all(nn_out == 0, axis=(0, 1))]
nn_out = nn_out[:, :, [0, 1, 2, 3, 7, 10, 11]] 
# reshape to (num_samples, num_features)
nn_in = nn_in.reshape(-1,nn_in.shape[2])
nn_out = nn_out.reshape(-1,nn_out.shape[2])
# shuffle the data
shuffled_indices = np.random.permutation(nn_in.shape[0])
nn_in = nn_in[shuffled_indices]
nn_out = nn_out[shuffled_indices]

print(f'nn_in shape: {nn_in.shape}')  # (num_samples, num_features)
print(f'nn_out shape: {nn_out.shape}')      # (num_samples, num_outputs)


nn_in shape: (658000, 16)
nn_out shape: (658000, 7)


In [14]:
# MODEL STRUCTURE

import torch
import torch.nn as nn
import torch.nn.functional as F

n_neurons = 512
class ThrustModel(nn.Module):
    def __init__(self, in_size=1, out_size=3):  
        super(ThrustModel, self).__init__()
        # Fully connected layers
        self.fc1 = nn.Linear(in_size, n_neurons) 
        self.fc2 = nn.Linear(n_neurons, n_neurons)
        self.fc3 = nn.Linear(n_neurons, n_neurons)
        self.fc4 = nn.Linear(n_neurons, n_neurons)
        self.fc5 = nn.Linear(n_neurons, n_neurons)
        self.fc6 = nn.Linear(n_neurons, n_neurons)
        self.fc7 = nn.Linear(n_neurons, n_neurons)
        self.fc8 = nn.Linear(n_neurons, n_neurons)
        self.fc9 = nn.Linear(n_neurons, n_neurons)
        self.output = nn.Linear(n_neurons, out_size)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc4(x))
        x = self.dropout(x)
        x = F.relu(self.fc5(x))
        x = self.dropout(x)
        x = F.relu(self.fc6(x))
        x = self.dropout(x)
        x = F.relu(self.fc7(x))
        x = self.dropout(x)
        x = F.relu(self.fc8(x))
        x = self.dropout(x)
        x = F.relu(self.fc9(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

model = ThrustModel(nn_in.shape[1], nn_out.shape[1])

# Print model summary
print(model)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

ThrustModel(
  (fc1): Linear(in_features=16, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=512, bias=True)
  (fc4): Linear(in_features=512, out_features=512, bias=True)
  (fc5): Linear(in_features=512, out_features=512, bias=True)
  (fc6): Linear(in_features=512, out_features=512, bias=True)
  (fc7): Linear(in_features=512, out_features=512, bias=True)
  (fc8): Linear(in_features=512, out_features=512, bias=True)
  (fc9): Linear(in_features=512, out_features=512, bias=True)
  (output): Linear(in_features=512, out_features=7, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
Total trainable parameters: 2113543


In [15]:
# load thrust model
model.load_state_dict(torch.load('thrust_model_v2_0.1306980233.pt'))

<All keys matched successfully>

In [20]:
# One step training
import torch.optim as optim
from torcheval.metrics.functional import mean_squared_error, r2_score
from scipy.integrate import odeint

def one_eval_training(model, input_data, target, r2_multiout=False):
    loss = 0.0
    model.eval()

    for i in range(target.shape[0]-1):

        with torch.no_grad():
            # Forward pass
            pred = model(input_data)  # Integrate over a small time step
    mse = mean_squared_error(pred, target)
            # Backward and optimize
    if r2_multiout:
        r2 = r2_score(pred, target, multioutput='raw_values')
    else:    
        r2 = r2_score(pred, target)
    return pred, mse, r2

In [34]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
pred, mse, r2 = one_eval_training(model, torch.tensor(nn_in[:1000], dtype=torch.float32).to(device), torch.tensor(nn_out[:1000], dtype=torch.float32).to(device), r2_multiout=True)
print(f'MSE: {mse}', f'R2: {r2}')
# Print real vs predicted for first 10 samples
print("Real vs Predicted (first 10 samples):")
for i in range(10):
    print(f"Real: \n{nn_out[i]}, \nPredicted: \n{pred[i].cpu().numpy()}")

MSE: 0.08866528421640396 R2: tensor([ 0.6143, -1.5785,  0.4395, -0.2069,  0.0044,  0.6835, -0.5796],
       device='cuda:0')
Real vs Predicted (first 10 samples):
Real: 
[ 0.00248959 -0.14159973 -0.00351878 -0.19990174 -0.14480107 -0.00197052
  0.01425545], 
Predicted: 
[ 0.39242244 -0.22952738  0.00481083 -0.38387597 -0.12906265  0.00260935
 -0.00110594]
Real: 
[ 0.10903206 -0.08554769 -0.15410554  0.24382945  0.17015805 -0.0862991
  0.01193607], 
Predicted: 
[ 0.23376137 -0.25341052  0.00297245  0.40929025  0.12401493  0.00168095
 -0.00251718]
Real: 
[ 0.60942825 -0.25279896 -0.10045312  0.43517243  0.13230823 -0.05625375
  0.02528984], 
Predicted: 
[ 0.3022347  -0.22869304  0.00381944  0.35906637  0.11116353  0.00214393
 -0.00225238]
Real: 
[ 0.21816663 -0.24050499 -0.10772992  0.41214061  0.15738746 -0.06032876
  0.06498853], 
Predicted: 
[ 0.17886186 -0.22278681  0.00226158  0.35110208  0.10244554  0.00126614
 -0.0020959 ]
Real: 
[ 0.23613519 -0.24427091 -0.07345751 -0.33592016 -0